<a href="https://colab.research.google.com/github/jetsonmom/inchang_lib_yolo/blob/main/%EC%9D%B8%EC%B0%BD%EB%8F%84%EC%84%9C%EA%B4%80_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

2. 유튜브 다운로드 문제 해결

In [ ]:
!pip install -U yt-dlp

3.  내 컴퓨터에서 영상 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()

video_path = next(iter(uploaded.keys()))

print("영상 파일:", video_path)

4. Ultralytics 설치

In [ ]:
!pip install -U ultralytics

5. 라이브러리 불러오기

In [ ]:
from ultralytics import YOLO
from collections import defaultdict

import os
import glob

from IPython.display import Video, display

6. 영상과 모델 경로 확인

In [ ]:
video_path = "/content/20261106_180533.mp4"
model_path = "/content/yolo11n.pt"

print("영상 파일:", os.path.exists(video_path))
print("모델 파일:", os.path.exists(model_path))

7. YOLOv11 모델 불러오기

In [ ]:
model = YOLO(model_path)

print("YOLO 모델 준비 완료!")

8. 객체 인식 및 추적

In [ ]:
# 물체별 분석 결과 저장
object_counts = defaultdict(int)
object_ids = defaultdict(set)

# YOLO 객체 추적 실행
results = model.track(
    source=video_path,
    tracker="bytetrack.yaml",
    persist=True,
    stream=True,
    save=True,
    show=False,
    project="/content/runs",
    name="all_objects",
    exist_ok=True
)

# 프레임별 분석
for result in results:

    if result.boxes is None:
        continue

    boxes = result.boxes

    if boxes.id is None:
        continue

    classes = boxes.cls.cpu().tolist()
    ids = boxes.id.cpu().tolist()

    for cls, track_id in zip(classes, ids):

        name = model.names[int(cls)]

        object_counts[name] += 1
        object_ids[name].add(int(track_id))

print("객체 인식 및 추적 완료!")

셀 ⑥ 분석 결과 출력

In [ ]:
print("===== 객체 분석 결과 =====")

for name in sorted(object_counts):

    print(f"\n물체 종류: {name}")
    print(f"프레임별 검출 횟수: {object_counts[name]}")
    print(f"고유 추적 개수: {len(object_ids[name])}")

셀 ⑦ 결과 영상 확인

In [ ]:
import glob
from IPython.display import Video

src = glob.glob("/content/runs/all_objects/20261106_180533.avi")[0]
dst = "/content/result_h264.mp4"

!ffmpeg -y -i "{src}" -vcodec libx264 -pix_fmt yuv420p -an "{dst}" -loglevel error

display(Video(dst, embed=True, width=640))